# Hof Co-op: planet population analysis (add-planets)

Analysis of runs made with `run_model_Hof.py` and `config/popsynth/DiscConfig_Hof_add-planets.json` (planets with `Mp = "SI"` are inserted where planetesimals form, then grown and migrated with the Bitsch model).

Units: `t` in the HDF5 file is in **yr** (shown here in Myr); `Rp` in AU; `Mcs`, `Mes` in M⊕.

In [ ]:
# ==== IMPORTS AND DIRECTORIES ====
# (Hof, 2026-11-11) Set-up cell for the planet-population ("add-planets") analysis.
# Runs come from run_model_Hof.py with config/popsynth/DiscConfig_Hof_add-planets.json.
import os
import re

import h5py
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

# Directory holding the add-planets pilot output (.h5). The run is written by
# run_model_Hof.py; per-planet series are HDF5 groups Mcs/<ip>, Mes/<ip>, Rp/<ip>
# with one row per entry of the dataset "t" [yr] (see run_model_Hof.py,
# create_planet_datasets() / write_planet_row()).
DATADIR_PILOT = "/home/imontesd/scratch/output/DiscEvolution/popsynth/add-planets/pilot_v1"

# Where figures are saved (None = do not save). Created on first save.
FIGDIR = None

### Planet evolution tracks (pilot v1)

Total mass (core + envelope, M⊕) against orbital radius (AU) for every planet in one pilot run. Same style as the *Planet evolution tracks* cell in `HJpaper.ipynb` (`compare_models_onepanel`):
- each line is one planet, coloured by its starting radius;
- the open circle is where the planet starts (its insertion point for `"SI"` planets);
- filled circles mark every 0.33 Myr, more opaque as time goes on.

This plot shows one model only. It uses a log colour scale from 0.2 to 30 AU (HJpaper used a linear 0–30 AU scale).

In [ ]:
# ==== PLANET EVOLUTION TRACKS (single run) ====
# (Hof, 2026-11-11) Total-mass growth tracks M_core + M_env [Mearth] vs orbital
# radius R_p [AU] for every planet in ONE run_model_Hof.py output file.
# Modelled on compare_models_onepanel() in HJpaper.ipynb ("Planet evolution tracks"
# cell, the paper's Fig. 6), with the same conventions:
#   * one line per planet, coloured by its starting radius ("Initial distance (AU)"
#     colour bar, cmap "plasma_r");
#   * circle markers every TIME_MARKER_INTERVAL_MYR [Myr], whose fill opacity grows
#     with time (alpha = t_mark / t_max), so you can read how fast each planet moved;
#   * the starting point is an OPEN circle.
# Differences from HJpaper (needed for this run):
#   1. One model per panel (HJpaper overlays two models, solid vs dashed).
#   2. Planets with Mp = "SI" are inserted only when planetesimals form at their
#      radius (streaming instability), so they may appear AFTER t = 0. run_model_Hof.py
#      backfills their rows with NaN until then (see _make_backfilled()). Here:
#        - the start point / colour comes from the Rp/<ip> attribute R_insert_au [AU]
#          (HJpaper uses Rp[ip][0], which would be NaN for a late insertion);
#        - the open circle sits at the INSERTION point (= t = 0 for planets present
#          from the start), and time markers are drawn only after insertion.
#   3. Colour scale is LOGARITHMIC between 0.2 and 30 AU (the planets.Rp range in
#      DiscConfig_Hof_add-planets.json). HJpaper used a linear 0-30 AU scale; with
#      8 of 16 radii inside 3 AU a linear scale would make those lines look alike.
#   4. A planet that the code stops writing early (e.g. removed at the inner edge)
#      has a shorter series than "t"; time is truncated to its length.
# Units: t [yr] in the file -> Myr here (/1e6); Rp [AU]; Mcs, Mes [Mearth].

PILOT_FILE = os.path.join(
    DATADIR_PILOT,
    "hof_add-planets_pilotv1_gamma1_psi476.0_erad0.967521_Ms0.977_aSS1.0e-05"
    "_Mdot7.4e-08_M7.7e-02_Rd9.5e+01.h5",
)

TIME_MARKER_INTERVAL_MYR = 0.33   # Myr between time markers (same as HJpaper)
R_COLOR_LIM_AU = (0.2, 30.0)      # AU, colour-bar range (log scale), see note 3


def load_planet_tracks_Hof(fname):
    """
    Read the per-planet time series of one run_model_Hof.py output file.

    Returns a dict:
        t_yr    : (nt,) time of each planet row [yr]  (dataset "t")
        planets : list (HDF5 index order = insertion order) of dicts with
                  ip            HDF5 index (str key of the Rp group)
                  config_index  position in config planets.Rp / planets.Mp (-1 if absent)
                  R_insert_au   insertion radius [AU]
                  t_form_Myr    insertion time [Myr]
                  Rp_au         (n,) orbital radius [AU]   (NaN before insertion)
                  Mc_Mearth     (n,) core mass [Mearth]
                  Me_Mearth     (n,) envelope mass [Mearth]
        params  : run parameters parsed from the filename (psi, erad, Ms [Msun],
                  aSS, Mdot [Msun/yr], M [Msun], Rd [AU]); missing tokens -> NaN
        complete: h5 attribute "complete" (False if the run did not finish)
    """
    out = {"planets": [], "params": {}}
    base = os.path.basename(fname)
    # Filename tokens written by run_model_Hof.py::output_filename()
    for key, pat in [("psi", r"_psi([0-9.eE+-]+?)_"), ("erad", r"_erad([0-9.eE+-]+?)_"),
                     ("Ms", r"_Ms([0-9.eE+-]+?)_"), ("aSS", r"_aSS([0-9.eE+-]+?)_Mdot"),
                     ("Mdot", r"_Mdot([0-9.eE+-]+?)_M"), ("M", r"_M([0-9.eE+-]+?)_Rd"),
                     ("Rd", r"_Rd([0-9.eE+-]+?)\.h5$")]:
        m = re.search(pat, base)
        out["params"][key] = float(m.group(1)) if m else np.nan

    with h5py.File(fname, "r") as f:
        out["t_yr"] = np.asarray(f["t"][()], dtype=float)                 # yr
        out["complete"] = bool(f.attrs.get("complete", False))
        if "Rp" not in f or len(f["Rp"].keys()) == 0:
            print(f"No planets in {base} (include_planets off, or none inserted yet).")
            return out
        for ip in sorted(f["Rp"].keys(), key=int):
            Rp = np.asarray(f["Rp"][ip][()], dtype=float)                 # AU
            Mc = np.asarray(f["Mcs"][ip][()], dtype=float)                # Mearth
            Me = np.asarray(f["Mes"][ip][()], dtype=float)                # Mearth
            a = f["Rp"][ip].attrs
            finite = np.isfinite(Rp)
            # Fallbacks for files written before the attributes existed:
            # first finite radius / time of the first finite row.
            R_ins = float(a["R_insert_au"]) if "R_insert_au" in a else (
                Rp[finite][0] if finite.any() else np.nan)
            t_ins_yr = float(a["t_form_yr"]) if "t_form_yr" in a else (
                out["t_yr"][np.argmax(finite)] if finite.any() else np.nan)
            out["planets"].append({
                "ip": ip,
                "config_index": int(a.get("config_index", -1)),
                "R_insert_au": R_ins,
                "t_form_Myr": t_ins_yr / 1e6,                              # yr -> Myr
                "Rp_au": Rp, "Mc_Mearth": Mc, "Me_Mearth": Me,
            })
    return out


def plot_planet_tracks_Hof(fname, planets=None, cmap="plasma_r",
                           time_marker_interval=TIME_MARKER_INTERVAL_MYR,
                           r_color_lim=R_COLOR_LIM_AU, xlim=None, ylim=None,
                           figdir=None):
    """
    Plot total mass (M_core + M_env) [Mearth] vs R_p [AU] for each planet of one run.
        planets : iterable of HDF5 planet indices (int) to draw; None = all
        figdir  : directory to save a PNG into (None = do not save)
    Returns (fig, ax, data) where data is the load_planet_tracks_Hof() dict.
    """
    data = load_planet_tracks_Hof(fname)
    if not data["planets"]:
        return None, None, data
    t_all_Myr = data["t_yr"] / 1e6                                         # yr -> Myr
    t_max = np.nanmax(t_all_Myr)                                           # Myr, end of run
    sel = [p for p in data["planets"] if planets is None or int(p["ip"]) in planets]

    norm = mpl.colors.LogNorm(vmin=r_color_lim[0], vmax=r_color_lim[1])   # AU
    cmap_obj = plt.get_cmap(cmap)

    fig, ax = plt.subplots()
    for p in sel:
        c = cmap_obj(norm(p["R_insert_au"]))
        n = len(p["Rp_au"])
        t_Myr = t_all_Myr[:n]                                              # note 4
        x = p["Rp_au"]                                                     # AU
        y = p["Mc_Mearth"] + p["Me_Mearth"]                                # Mearth
        ok = np.isfinite(x) & np.isfinite(y) & (y > 0)                     # drop NaN backfill
        if not ok.any():
            continue
        ax.loglog(x[ok], y[ok], color=c, linestyle="-", alpha=0.9)

        # Open circle at the insertion point (HJpaper: open circle at t = 0)
        i0 = np.argmax(ok)
        ax.scatter(x[i0], y[i0], facecolors="none", edgecolors=c, s=100,
                   marker="o", linewidths=1.5, zorder=5)

        # Filled markers every time_marker_interval [Myr] after insertion, with fill
        # opacity = t_mark / t_max (HJpaper's "progressive filling")
        for t_mark in np.arange(time_marker_interval, t_max, time_marker_interval):
            if t_mark < t_Myr[i0] or t_mark > t_Myr[ok][-1]:
                continue                       # planet not yet inserted / no longer written
            idx = np.argmin(np.abs(t_Myr - t_mark))
            if not ok[idx]:
                continue
            ax.scatter(x[idx], y[idx], color=c, edgecolors=c, s=100, marker="o",
                       alpha=t_mark / t_max, linewidths=1.5, zorder=5)

    ax.set_xlabel("Distance (AU)")
    ax.set_ylabel("Total mass ($M_⊕$)")
    if xlim: ax.set_xlim(xlim)
    if ylim: ax.set_ylim(ylim)

    # Short run label: wind strength psi (dimensionless), M* [Msun], initial disc
    # mass M [Msun] and characteristic radius Rd [AU], from the filename
    pr = data["params"]
    ax.set_title(rf"$\psi$={pr['psi']:g}, $M_*$={pr['Ms']:g} $M_\odot$, "
                 rf"$M_d$={pr['M']:.2g} $M_\odot$, $R_d$={pr['Rd']:g} au"
                 + ("" if data["complete"] else "  [run incomplete]"), fontsize=10)

    sm = mpl.cm.ScalarMappable(cmap=cmap_obj, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("Initial distance (AU)")
    plt.tight_layout()

    if figdir:
        os.makedirs(figdir, exist_ok=True)
        outname = os.path.join(figdir, os.path.basename(fname).replace(".h5", "_tracks_Mtot_vs_R.png"))
        fig.savefig(outname, dpi=200)
        print(f"Saved {outname}")
    return fig, ax, data


fig, ax, tracks_pilot = plot_planet_tracks_Hof(PILOT_FILE, figdir=FIGDIR)
plt.show()

# Summary table: one line per planet (HDF5 insertion order)
print(f"t_end = {tracks_pilot['t_yr'][-1] / 1e6:.3f} Myr, complete = {tracks_pilot['complete']}, "
      f"N planets = {len(tracks_pilot['planets'])}")
print(f"{'ip':>3} {'cfg':>4} {'R_ins[AU]':>9} {'t_form[Myr]':>11} {'R_fin[AU]':>9} "
      f"{'Mc[ME]':>9} {'Me[ME]':>9} {'Mtot[ME]':>9}")
for p in tracks_pilot["planets"]:
    last = np.flatnonzero(np.isfinite(p["Rp_au"]))
    j = last[-1] if last.size else -1
    print(f"{p['ip']:>3} {p['config_index']:>4} {p['R_insert_au']:9.3f} {p['t_form_Myr']:11.3f} "
          f"{p['Rp_au'][j]:9.3f} {p['Mc_Mearth'][j]:9.3g} {p['Me_Mearth'][j]:9.3g} "
          f"{p['Mc_Mearth'][j] + p['Me_Mearth'][j]:9.3g}")

### Planet total mass vs time (pilot v1)

Total mass (core + envelope, M⊕) against time (Myr) for the same planets as the tracks plot above, with the same colour coding (starting radius, log scale 0.2–30 AU).
- An open circle marks insertion: t = 0, or the insertion time for `"SI"` planets.
- A filled circle marks the last saved point.

The time axis is linear by default; call with `logt=True` for a log axis, which shows the early growth more clearly. This cell reuses `tracks_pilot` from the cell above, so run that cell first.

In [ ]:
# ==== PLANET TOTAL MASS vs TIME (single run) ====
# (Hof, 2026-11-11) Companion to the tracks cell above: total mass
# M_core + M_env [Mearth] against time t [Myr] for every planet of the same run.
# HJpaper.ipynb has no mass-vs-time panel, so this keeps the conventions of the
# tracks cell (and of HJpaper's compare_models_onepanel) instead:
#   * one line per planet, coloured by insertion radius R_insert_au [AU], with the
#     same log colour scale (R_COLOR_LIM_AU) and "Initial distance (AU)" colour bar;
#   * OPEN circle at the insertion point (t_form; t = 0 for planets present at start);
#   * FILLED circle at the last written row (final mass, or the moment the code
#     stopped writing the planet).
# The 0.33 Myr time markers of the tracks plot are dropped here: with time on the
# x-axis they would only repeat the axis ticks.
# Time axis: LINEAR by default (0 to t_end), so the insertion times of "SI" planets
# are easy to read. Set logt=True for a log time axis (resolves the fast early
# growth); rows with t = 0 are then dropped, since log(0) is undefined.
# Uses tracks_pilot (the load_planet_tracks_Hof() dict from the cell above), so the
# file is not read twice. Units: t [yr] in the file -> Myr (/1e6); masses in Mearth.


def plot_planet_mass_vs_time_Hof(data, planets=None, cmap="plasma_r",
                                 r_color_lim=R_COLOR_LIM_AU, logt=False,
                                 xlim=None, ylim=None, figdir=None, fname=None):
    """
    Plot total mass (M_core + M_env) [Mearth] vs time [Myr] for each planet of one run.
        data    : dict from load_planet_tracks_Hof()
        planets : iterable of HDF5 planet indices (int) to draw; None = all
        logt    : False = linear time axis (default), True = log time axis
        figdir  : directory to save a PNG into (None = do not save); needs fname
                  (the .h5 path) to build the PNG name
    Returns (fig, ax).
    """
    if not data["planets"]:
        print("No planets to plot.")
        return None, None
    t_all_Myr = data["t_yr"] / 1e6                                         # yr -> Myr
    sel = [p for p in data["planets"] if planets is None or int(p["ip"]) in planets]

    norm = mpl.colors.LogNorm(vmin=r_color_lim[0], vmax=r_color_lim[1])   # AU
    cmap_obj = plt.get_cmap(cmap)

    fig, ax = plt.subplots()
    plotfunc = ax.loglog if logt else ax.semilogy
    for p in sel:
        c = cmap_obj(norm(p["R_insert_au"]))
        n = len(p["Mc_Mearth"])
        t_Myr = t_all_Myr[:n]                      # truncate to this planet's rows (Myr)
        y = p["Mc_Mearth"] + p["Me_Mearth"]        # Mearth
        ok = np.isfinite(y) & (y > 0)              # drop NaN backfill before insertion
        if logt:
            ok &= t_Myr > 0                        # log axis cannot show t = 0
        if not ok.any():
            continue
        plotfunc(t_Myr[ok], y[ok], color=c, linestyle="-", alpha=0.9)
        i0, i1 = np.flatnonzero(ok)[[0, -1]]       # first / last plotted row
        ax.scatter(t_Myr[i0], y[i0], facecolors="none", edgecolors=c, s=100,
                   marker="o", linewidths=1.5, zorder=5)     # insertion
        ax.scatter(t_Myr[i1], y[i1], color=c, edgecolors=c, s=100,
                   marker="o", linewidths=1.5, zorder=5)     # last written row

    ax.set_xlabel("Time (Myr)")
    ax.set_ylabel("Total mass ($M_⊕$)")
    if xlim:
        ax.set_xlim(xlim)
    elif not logt:
        # Myr, start of run to t_end, plus a 2% margin so the final-mass circles
        # at t_end are not cut in half by the axis edge
        ax.set_xlim(0, 1.02 * np.nanmax(t_all_Myr))
    if ylim: ax.set_ylim(ylim)

    # Same run label as the tracks plot (psi dimensionless, M* and M_d in Msun, R_d in au)
    pr = data["params"]
    ax.set_title(rf"$\psi$={pr['psi']:g}, $M_*$={pr['Ms']:g} $M_\odot$, "
                 rf"$M_d$={pr['M']:.2g} $M_\odot$, $R_d$={pr['Rd']:g} au"
                 + ("" if data["complete"] else "  [run incomplete]"), fontsize=10)

    sm = mpl.cm.ScalarMappable(cmap=cmap_obj, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("Initial distance (AU)")
    plt.tight_layout()

    if figdir and fname:
        os.makedirs(figdir, exist_ok=True)
        tag = "_Mtot_vs_logt.png" if logt else "_Mtot_vs_t.png"
        outname = os.path.join(figdir, os.path.basename(fname).replace(".h5", tag))
        fig.savefig(outname, dpi=200)
        print(f"Saved {outname}")
    return fig, ax


fig, ax = plot_planet_mass_vs_time_Hof(tracks_pilot, figdir=FIGDIR, fname=PILOT_FILE)
plt.show()